# Deepnote Reviewer via Greft

This notebook receives an experiment result sent from Google Colab through Greft, applies a deterministic quality rule, and sends the review back to Colab.

The review rule is:

```text
PASS if accuracy >= 0.90
otherwise REVIEW
```


## 1. Install dependencies

In [2]:
%pip install -q "mcp" "httpx2"



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Load Deepnote environment variables

Create a Deepnote **Environment variables** integration containing `GREFT_API_URL` and `GREFT_API_KEY`, and connect it to this project before running the cell.


In [23]:
import os
import json
import httpx2

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

GREFT_API_URL = os.environ["GREFT_API_URL"].rstrip("/")
GREFT_API_KEY = os.environ["GREFT_API_KEY"]

DEEPNOTE_ADDRESS = "@reviewer"
COLAB_ADDRESS = "@ralph-planner"

print("Deepnote configuration loaded.")


Deepnote configuration loaded.


## 3. Greft MCP helper

The message selector scans the mailbox from newest to oldest and ignores unrelated non-JSON messages.


In [25]:
async def call_greft_tool(address, tool_name, arguments=None):
    mcp_url = f"{GREFT_API_URL}/mcp?address={address}"

    async with httpx2.AsyncClient(
        headers={"Authorization": f"Bearer {GREFT_API_KEY}"},
        timeout=httpx2.Timeout(30.0, read=300.0),
    ) as http_client:
        async with streamable_http_client(
            mcp_url,
            http_client=http_client,
        ) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                return await session.call_tool(
                    tool_name,
                    arguments or {},
                )


def get_payload_text(message):
    payload = message.get("payload") or {}

    return (
        payload.get("text")
        or payload.get("message")
        or payload.get("body")
    )


def find_latest_json_message(messages, kind, experiment_id=None):
    ordered = sorted(
        messages,
        key=lambda message: message.get("seq", -1),
        reverse=True,
    )

    for message in ordered:
        text = get_payload_text(message)

        if not isinstance(text, str):
            continue

        try:
            data = json.loads(text)
        except json.JSONDecodeError:
            continue

        if data.get("kind") != kind:
            continue

        if (
            experiment_id is not None
            and data.get("experiment_id") != experiment_id
        ):
            continue

        return message, data

    return None, None


## 4. Read the latest Colab experiment

Run this cell **after Colab sends the experiment**. Using `read_messages` avoids accidentally receiving an older unacknowledged email or test message.


In [27]:
result = await call_greft_tool(
    DEEPNOTE_ADDRESS,
    "read_messages",
    {},
)


messages = (result.structured_content or {}).get("messages", [])

experiment_message, experiment = find_latest_json_message(
    messages,
    kind="experiment_result",
)

if experiment is None:
    raise RuntimeError(
        "No experiment_result message was found. "
        "Send the experiment from Colab, then run this cell again."
    )

print("Received experiment:")
print(json.dumps(experiment, indent=2))


meta=None content=[TextContent(type='text', text='{"messages":[{"id":"msg_01M2J0PFTVE8NAYBJJJTQKC00F","seq":92,"conversation_id":"conv_01M2J0PFMGT8QJYZN3GAJHTMK2","from_agent_id":"agt_01M2G9GXZ0J4295GHF7ADJWGT3","to_agent_id":"agt_01M27QVH6X6NRDE5V4YJCTN8JW","type":"request","payload":{"html":"<div dir=3D\\"ltr\\">Hi</div><br><div class=3D\\"gmail_quote gmail_quote_containe=\\nr\\"><div dir=3D\\"ltr\\" class=3D\\"gmail_attr\\">On Tue, 15 Sept 2026 at 08:49, Gre=\\nft Agent &lt;<a href=3D\\"mailto:reviewer@greft.ai\\">reviewer@greft.ai</a>&gt;=\\n wrote:<br></div><blockquote class=3D\\"gmail_quote\\" style=3D\\"margin:0px 0px =\\n0px 0.8ex;border-left:1px solid rgb(204,204,204);padding-left:1ex\\"><p>Hello=\\n</p></blockquote></div>","text":"Hi\\n\\nOn Tue, 15 Sept 2026 at 08:49, Greft Agent <reviewer@greft.ai> wrote:\\n\\n> Hello\\n>","subject":"Re: Greft email test","to_email":"reviewer@greft.ai","from_email":"ralphnjambou@gmail.com"},"metadata":{"subject":"Re: Greft email test","prov

## 5. Review the experiment

In [29]:
MINIMUM_ACCURACY = 0.90

accuracy = float(experiment["accuracy"])

review = {
    "kind": "experiment_review",
    "experiment_id": experiment["experiment_id"],
    "verdict": "PASS" if accuracy >= MINIMUM_ACCURACY else "REVIEW",
    "accuracy": accuracy,
    "required_accuracy": MINIMUM_ACCURACY,
}

print(json.dumps(review, indent=2))


{
  "kind": "experiment_review",
  "experiment_id": "iris-logreg-b4452505",
  "verdict": "PASS",
  "accuracy": 0.9667,
  "required_accuracy": 0.9
}


## 6. Send the review back to Colab

After this cell succeeds, return to `colab.ipynb` and run its final inbox cell.


In [31]:
await call_greft_tool(
    DEEPNOTE_ADDRESS,
    "send_message",
    {
        "to": COLAB_ADDRESS,
        "message": json.dumps(review),
    },
)

print(
    f"Sent {review['verdict']} review for "
    f"{review['experiment_id']} to {COLAB_ADDRESS}."
)


Sent PASS review for iris-logreg-b4452505 to @ralph-planner.


## Expected behavior

The notebook should receive the current `experiment_result`, produce either `PASS` or `REVIEW`, and send an `experiment_review` carrying the same `experiment_id` back through Greft.

Before committing this notebook, clear outputs containing real account-specific IDs or messages.


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=517e1466-901c-4fec-bb9a-7dc9939b5f98' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>